# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
!pip install -q huggingface_hub datasets pandas pyarrow scikit-learn

from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MONTH = "2026-03"

try:
    ds = load_dataset(
        "FlyRank/internship-warehouse",
        data_files=f"fact_content_daily_performance/month={MONTH}/*.parquet",
        split="train",
        token=HF_TOKEN,
    )
    df_month = ds.to_pandas()
    print(f"Fast path worked — loaded {len(df_month):,} rows for {MONTH}.")
except Exception as e:
    print("Fast path failed, falling back to streaming (this takes a few minutes).")
    print("Reason:", e)
    ds = load_dataset(
        "FlyRank/internship-warehouse",
        "fact_content_daily_performance",
        split="train",
        streaming=True,
        token=HF_TOKEN,
    )
    rows = [row for row in ds if str(row["report_date"]).startswith(MONTH)]
    df_month = pd.DataFrame(rows)
    print(f"Streaming path loaded {len(df_month):,} rows for {MONTH}.")

print(df_month.shape)
print(df_month["report_date"].min(), "to", df_month["report_date"].max())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Fast path worked — loaded 9,841,378 rows for 2026-03.
(9841378, 30)
2026-03-01 to 2026-03-31


## 2. Fields: feature / label / context / excluded

***Feature** (all knowable before March 16, the decision moment):
- gsc_impressions, gsc_clicks summed over March 1–15
- ctr_first_half, computed from those same first-half sums
- gsc_avg_position averaged over March 1–15
- active_days_first_half — count of first-half days with gsc_data_available = True

**Label/proxy:** is_declining_proxy — 1 if clicks in the second half of March are lower
than the first half, else 0. A real, observed within-month trend.

**Context:** client_hash_id, content_hash_id — pseudonyms, used only for grouping,
never as features.

**Excluded:** fact_content_query_90d — its 90-day window overlaps this month's label
window, so anything from it risks leaking future information into a feature.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- Query 1: grain check ---
grain_check = df_month.groupby(["report_date", "client_hash_id", "content_hash_id"]).size()
dupes = grain_check[grain_check > 1]
print(f"Rows: {len(df_month):,} | Duplicate (date, client, content) combos: {len(dupes)}")
print("Grain holds." if len(dupes) == 0 else "Grain broken — investigate.")

# --- Query 2: counts + date span ---
print("\nRow count:", len(df_month))
print("Date span:", df_month["report_date"].min(), "to", df_month["report_date"].max())
print("Unique clients:", df_month["client_hash_id"].nunique())
print("Unique content items:", df_month["content_hash_id"].nunique())

# --- Query 3: availability filter (IS TRUE) ---
before = len(df_month)
avail = df_month[df_month["gsc_data_available"] == True]
after = len(avail)
print(f"\nRows before gsc_data_available filter: {before:,}")
print(f"Rows after gsc_data_available IS TRUE: {after:,} ({after/before*100:.1f}%)")

# --- FIX: keep only real, available rows before aggregating ---
# (previously this step was missing, so zero-filled placeholder rows
# were being treated as real zero-activity days — corrected here)
df_month = df_month[df_month["gsc_data_available"] == True].copy()

# --- Build first-half / second-half split (past = features, future = label) ---
df_month["report_date"] = pd.to_datetime(df_month["report_date"])
first_half = df_month[df_month["report_date"] <= "2026-03-15"]
second_half = df_month[df_month["report_date"] >= "2026-03-16"]
group_cols = ["client_hash_id", "content_hash_id"]

first_agg = first_half.groupby(group_cols).agg(
    impressions_first_half=("gsc_impressions", "sum"),
    clicks_first_half=("gsc_clicks", "sum"),
    avg_position_first_half=("gsc_avg_position", "mean"),
    active_days_first_half=("gsc_data_available", "sum"),
).reset_index()

second_agg = second_half.groupby(group_cols).agg(
    clicks_second_half=("gsc_clicks", "sum"),
).reset_index()

pair_df = first_agg.merge(second_agg, on=group_cols, how="inner")
pair_df["ctr_first_half"] = (pair_df["clicks_first_half"] / pair_df["impressions_first_half"].replace(0, np.nan)) * 100
pair_df["is_declining_proxy"] = (pair_df["clicks_second_half"] < pair_df["clicks_first_half"]).astype(int)

print(f"\n5-feature frame: {pair_df.shape[0]:,} (client, content) pairs")
pair_df.head()

Rows: 9,841,378 | Duplicate (date, client, content) combos: 0
Grain holds.

Row count: 9841378
Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Unique clients: 55
Unique content items: 331437

Rows before gsc_data_available filter: 9,841,378
Rows after gsc_data_available IS TRUE: 3,611,061 (36.7%)

5-feature frame: 141,467 (client, content) pairs


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,active_days_first_half,clicks_second_half,ctr_first_half,is_declining_proxy
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119,1,12.639599,15,1,0.840336,0
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72,0,8.094074,15,0,0.000000,0
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,283,0,12.587226,15,0,0.000000,0
3,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9,0,11.500000,5,1,0.000000,0
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66,0,12.282875,13,0,0.000000,0


In [5]:
# --- 5 features, each "available when?" ---
print("1. impressions_first_half — available when? Fully in the past window (Mar 1-15).")
print("2. clicks_first_half — available when? Same, fully past.")
print("3. ctr_first_half — available when? Derived only from past-window sums.")
print("4. avg_position_first_half — available when? Averaged only over past days.")
print("5. active_days_first_half — available when? Count of past days only.")

1. impressions_first_half — available when? Fully in the past window (Mar 1-15).
2. clicks_first_half — available when? Same, fully past.
3. ctr_first_half — available when? Derived only from past-window sums.
4. avg_position_first_half — available when? Averaged only over past days.
5. active_days_first_half — available when? Count of past days only.


In [6]:
# --- THE TRAP: deliberately add a leaking feature ---
pair_df["clicks_full_month"] = pair_df["clicks_first_half"] + pair_df["clicks_second_half"]
# clicks_full_month LOOKS like a harmless "total engagement" feature, but it secretly
# contains clicks_second_half — the exact thing the label is computed from.

honest_features = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]
leaky_features = honest_features + ["clicks_full_month"]

X_honest = pair_df[honest_features].fillna(0)
X_leaky = pair_df[leaky_features].fillna(0)
y = pair_df["is_declining_proxy"]

Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
Xl_tr, Xl_te, _, _ = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)

honest_score = accuracy_score(y_te, LogisticRegression(max_iter=1000).fit(Xh_tr, y_tr).predict(Xh_te))
leaky_score = accuracy_score(y_te, LogisticRegression(max_iter=1000).fit(Xl_tr, y_tr).predict(Xl_te))

print(f"Honest quick score (5 features): {honest_score:.3f}")
print(f"LEAKY quick score (+clicks_full_month): {leaky_score:.3f}")
print("\nclicks_full_month jumps the score toward perfect because it bakes in second-half")
print("clicks — the same data the label is made from. Deleting it and keeping the honest")
print(f"score ({honest_score:.3f}) is the right call.")

Honest quick score (5 features): 0.922
LEAKY quick score (+clicks_full_month): 1.000

clicks_full_month jumps the score toward perfect because it bakes in second-half
clicks — the same data the label is made from. Deleting it and keeping the honest
score (0.922) is the right call.


## 4. Data limits

A second limitation, caught while building features: the daily table zero-fills rows
where gsc_data_available = False rather than omitting them. Before filtering, my
first-half/second-half aggregates included these placeholder zeros as if they were
real zero-activity days. Filtering to gsc_data_available = True before aggregating
(rather than only checking it as a standalone query) was necessary to avoid treating
"no data" as "no engagement."

In [7]:
dim_clients = load_dataset("FlyRank/internship-warehouse", "dim_clients", split="train", token=HF_TOKEN).to_pandas()
print(dim_clients["access_profile"].value_counts())
print(f"\n{(dim_clients['access_profile'] != 'gsc_and_ga4').mean()*100:.1f}% of clients lack full GSC+GA4 access.")

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

access_profile
gsc_and_ga4                             53
no_search_or_analytics_access           26
gsc_only                                14
source_only_missing_client_dimension    10
ga4_only                                 1
Name: count, dtype: int64

49.0% of clients lack full GSC+GA4 access.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.